# XDrift: Temporal Stability of Explainable AI in Credit Risk Models
**Research Implementation Notebook**

This notebook implements the full XDrift framework:
- **Phase 1** — Data pipeline & feature engineering (Lending Club)
- **Phase 2** — XGBoost credit default model with Optuna tuning
- **Phase 3** — XAI Engine: TreeSHAP (global + local) and LIME
- **Phase 4** — XDrift framework: rolling windows, XSI metric, HMM regime detection, EART trigger
- **Phase 5** — All paper figures and results tables

**Dataset:** Add the Lending Club dataset via Kaggle Datasets:
`wordsforthewise/lending-club` → attach as input to this notebook.

**Paper:** *XDrift: Monitoring and Mitigating Temporal Explanation Drift in Credit Default Prediction under Market Regime Shifts*


## Cell Cluster 1 — Environment Setup

In [ ]:
# Install libraries not pre-installed on Kaggle.
# pandas, numpy, sklearn, xgboost, shap, matplotlib, seaborn,
# scipy, optuna are all pre-installed. We only need these three.
!pip install -q hmmlearn lime optuna

import subprocess, sys
print("Installation complete.")


In [ ]:
# ── Core imports ──────────────────────────────────────────────────
import os, json, pickle, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
from tqdm.notebook import tqdm

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, classification_report,
    roc_curve, average_precision_score,
)
from sklearn.calibration import calibration_curve

# XGBoost + Optuna
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# XAI
import shap
from lime.lime_tabular import LimeTabularExplainer

# Stats + HMM
from scipy.stats import kendalltau, spearmanr, ks_2samp
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")

# ── Reproducibility seeds ──────────────────────────────────────
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
shap.initjs()
print("All imports successful.")

# ── Logging ───────────────────────────────────────────────────
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('xdrift')


In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — all hyperparameters and paths in one place.
# On Kaggle: input data lives in /kaggle/input/
#            all outputs go to /kaggle/working/
# ══════════════════════════════════════════════════════════════

# ── Paths ─────────────────────────────────────────────────────────
# Kaggle attaches the Lending Club dataset at this path.
# Dataset slug: wordsforthewise/lending-club
# File inside: accepted_2007_to_2018Q4.csv (or similar name)
RAW_DATA_FILE = "/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"
print(f"Dataset found: {RAW_DATA_FILE}")

WORKING_DIR    = "/kaggle/working"
FIGURES_DIR    = os.path.join(WORKING_DIR, "figures")
MODELS_DIR     = os.path.join(WORKING_DIR, "models")
RESULTS_DIR    = os.path.join(WORKING_DIR, "results")

for d in [FIGURES_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Column / target config ────────────────────────────────────────
TARGET_COL      = "default"     # binary: 1=default, 0=fully paid
DATE_COL        = "issue_d"     # loan issue date
TRAIN_END_DATE  = "2017-12-31"
TEST_START_DATE = "2018-01-01"

# ── XGBoost defaults (used if Optuna is skipped) ─────────────────
XGB_PARAMS = {
    "objective":        "binary:logistic",
    "eval_metric":      "auc",
    "n_estimators":     800,
    "max_depth":        6,
    "learning_rate":    0.05,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "gamma":            0.1,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "scale_pos_weight": 3,       # ~25% default rate → weight positives
    "random_state":     42,
    "tree_method":      "hist",  # fast GPU-compatible histogram method
    "device":           "cuda" if os.path.exists("/proc/driver/nvidia") else "cpu",
    "n_jobs":           -1,
    "enable_categorical": False,  # newer XGBoost defaults this to True, which breaks SHAP TreeExplainer
}
print(f"XGBoost device: {XGB_PARAMS['device']}")

# ── Rolling-window settings ───────────────────────────────────────
WINDOW_MONTHS   = 12    # each training window = 12 months
STRIDE_MONTHS   = 3     # slide forward 3 months per step
MIN_WINDOW_SIZE = 1000  # skip windows with fewer loans than this

# ── XSI thresholds ────────────────────────────────────────────────
XSI_ALERT_THRESHOLD    = 0.70   # EART fires when XSI drops below this
XSI_CRITICAL_THRESHOLD = 0.50   # human review flag
TOP_K_FEATURES         = 15     # features tracked in XSI

# ── HMM regime detection ──────────────────────────────────────────
N_REGIMES    = 3
REGIME_NAMES = {0: "Bull", 1: "Neutral", 2: "Bear/Crisis"}

# ── SHAP sampling ─────────────────────────────────────────────────
SHAP_BACKGROUND_SAMPLES = 500
SHAP_EXPLAIN_SAMPLES    = 1000
RANDOM_SEED             = 42

print("Configuration loaded.")

# ── Run metadata ──────────────────────────────────────────────────────
import hashlib, platform, datetime
RUN_METADATA = {
    "run_date":     datetime.datetime.now().isoformat(),
    "python":       platform.python_version(),
    "seed":         RANDOM_SEED,
    "window_months": WINDOW_MONTHS,
    "stride_months": STRIDE_MONTHS,
    "xsi_alert":    XSI_ALERT_THRESHOLD,
    "xsi_critical": XSI_CRITICAL_THRESHOLD,
    "top_k":        TOP_K_FEATURES,
}
print(f"Run metadata: {json.dumps(RUN_METADATA, indent=2)}")


## Cell Cluster 2 — Data Pipeline

Loads the Lending Club CSV, filters to resolved outcomes only (Fully Paid vs Charged Off/Default),
engineers financial features, merges quarterly macroeconomic context (Fed Funds Rate, VIX bucket),
encodes categoricals, and produces clean train/test splits.


In [ ]:
# ── Feature selection ────────────────────────────────────────────
# Only columns available at loan origination (no data leakage).
LOAN_FEATURES = [
    "loan_amnt", "funded_amnt", "int_rate", "installment",
    "term", "grade", "sub_grade", "purpose", "initial_list_status",
    "annual_inc", "emp_length", "home_ownership", "verification_status",
    "addr_state", "dti", "delinq_2yrs", "inq_last_6mths", "open_acc",
    "pub_rec", "revol_bal", "revol_util", "total_acc", "earliest_cr_line",
    "loan_status", "issue_d",
]

DEFAULT_STATUSES = {"Charged Off", "Default"}
PAID_STATUSES    = {"Fully Paid"}

# ── Macro context table ───────────────────────────────────────────
# Quarterly Fed Funds Rate + VIX bucket (0=low, 1=med, 2=high).
# This is hand-curated from Federal Reserve and CBOE historical data.
# It gives the model economic regime awareness at origination time.
MACRO_DATA = {
    (2007,1):(5.26,1),(2007,2):(5.25,1),(2007,3):(5.02,2),(2007,4):(4.24,2),
    (2008,1):(2.98,2),(2008,2):(2.00,2),(2008,3):(1.81,2),(2008,4):(0.16,2),
    (2009,1):(0.18,2),(2009,2):(0.18,2),(2009,3):(0.15,1),(2009,4):(0.12,1),
    (2010,1):(0.13,1),(2010,2):(0.18,1),(2010,3):(0.19,1),(2010,4):(0.19,1),
    (2011,1):(0.14,1),(2011,2):(0.09,1),(2011,3):(0.08,2),(2011,4):(0.07,2),
    (2012,1):(0.08,1),(2012,2):(0.16,1),(2012,3):(0.14,1),(2012,4):(0.16,1),
    (2013,1):(0.15,0),(2013,2):(0.11,0),(2013,3):(0.09,0),(2013,4):(0.09,0),
    (2014,1):(0.08,0),(2014,2):(0.10,0),(2014,3):(0.09,0),(2014,4):(0.12,0),
    (2015,1):(0.11,1),(2015,2):(0.13,1),(2015,3):(0.14,2),(2015,4):(0.24,1),
    (2016,1):(0.36,1),(2016,2):(0.37,1),(2016,3):(0.40,0),(2016,4):(0.55,0),
    (2017,1):(0.79,0),(2017,2):(1.04,0),(2017,3):(1.15,0),(2017,4):(1.30,0),
    (2018,1):(1.51,0),(2018,2):(1.82,1),(2018,3):(1.95,1),(2018,4):(2.27,1),
    (2019,1):(2.40,1),(2019,2):(2.38,1),(2019,3):(2.13,1),(2019,4):(1.80,0),
    (2020,1):(0.65,2),(2020,2):(0.08,2),(2020,3):(0.09,2),(2020,4):(0.09,2),
    (2021,1):(0.07,1),(2021,2):(0.06,1),(2021,3):(0.08,1),(2021,4):(0.08,1),
    (2022,1):(0.20,1),(2022,2):(1.21,2),(2022,3):(2.50,2),(2022,4):(3.78,2),
    (2023,1):(4.65,1),(2023,2):(5.08,1),(2023,3):(5.33,1),(2023,4):(5.33,1),
    (2024,1):(5.33,1),(2024,2):(5.33,1),(2024,3):(5.13,0),(2024,4):(4.60,0),
}

print("Pipeline constants defined.")


In [ ]:
# ── Parsing helpers ──────────────────────────────────────────────

def parse_employment_length(emp_str):
    """'10+ years' → 10.0, '< 1 year' → 0.5, '3 years' → 3.0"""
    if pd.isna(emp_str):
        return np.nan
    s = str(emp_str).strip().lower()
    if "10+" in s: return 10.0
    if "< 1" in s: return 0.5
    digits = "".join(c for c in s if c.isdigit() or c == ".")
    return float(digits) if digits else np.nan

def parse_revolving_util(val):
    """'62.5%' → 62.5"""
    if pd.isna(val): return np.nan
    return float(str(val).replace("%", "").strip())

# ── Macroeconomic feature merger ─────────────────────────────────

def add_macro_features(df):
    """
    Joins Fed Funds Rate and VIX bucket to each loan
    based on the quarter of its issue date.
    Gives the model economic context at origination time.
    """
    df = df.copy()
    df["issue_year"]    = df["issue_d"].dt.year
    df["issue_quarter"] = df["issue_d"].dt.quarter
    fed_rates, vix_buckets = [], []
    for _, row in df[["issue_year", "issue_quarter"]].iterrows():
        rate, vix = MACRO_DATA.get((int(row.issue_year), int(row.issue_quarter)), (np.nan, np.nan))
        fed_rates.append(rate)
        vix_buckets.append(vix)
    df["fed_funds_rate"] = fed_rates
    df["vix_bucket"]     = vix_buckets
    return df

# ── Feature engineering ───────────────────────────────────────────

def engineer_features(df):
    """
    Derives financial ratio features and log-transforms.
    All features are computed from origination-time data only.
    """
    df = df.copy()
    # Loan-to-income: higher = more financial stress
    df["loan_to_income"]    = df["loan_amnt"] / (df["annual_inc"] + 1)
    # Credit age: years between first credit line and loan issuance
    df["credit_age_years"]  = (
        df["issue_d"].dt.year -
        pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce").dt.year
    ).clip(0, 40)
    # Interaction: utilization x DTI = combined stress signal
    df["util_dti_interact"] = df["revol_util"] * df["dti"]
    # Log-transforms reduce right skew on monetary columns
    for col in ["annual_inc", "loan_amnt", "revol_bal", "installment"]:
        df[f"log_{col}"] = np.log1p(df[col].clip(0))
    # Ordinal grade: A=1 ... G=7
    grade_map = {"A":1,"B":2,"C":3,"D":4,"E":5,"F":6,"G":7}
    df["grade_num"]   = df["grade"].map(grade_map).fillna(4)
    # Binary flags
    df["has_delinq"]  = (df["delinq_2yrs"] > 0).astype(int)
    df["has_pub_rec"] = (df["pub_rec"]     > 0).astype(int)
    return df

# ── Categorical encoding ──────────────────────────────────────────

def encode_categoricals(df):
    """
    Frequency-encodes addr_state (high cardinality).
    Label-encodes all other object columns.
    """
    df = df.copy()
    state_freq = df["addr_state"].value_counts(normalize=True)
    df["addr_state_freq"] = df["addr_state"].map(state_freq).fillna(0)
    cat_cols = ["grade","sub_grade","purpose","home_ownership",
                "verification_status","initial_list_status","addr_state"]
    for col in cat_cols:
        if col == "addr_state":
            continue
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str).fillna("Unknown"))
    return df

# ── Feature column list ───────────────────────────────────────────

def get_feature_columns(df):
    """Everything except target, date, and helper date columns."""
    exclude = {TARGET_COL, DATE_COL, "issue_year", "issue_quarter"}
    return [c for c in df.columns if c not in exclude]

print("Pipeline functions defined.")


In [ ]:
# ── LOAD AND PROCESS DATA ─────────────────────────────────────────
print(f"Loading: {RAW_DATA_FILE}")
t0 = time.time()

print(f"Reading: {RAW_DATA_FILE}")
df_raw = pd.read_csv(RAW_DATA_FILE, usecols=LOAN_FEATURES, low_memory=False)
print(f"Raw shape: {df_raw.shape}")

# Keep only resolved loan outcomes (drop Current, Late, In Grace Period etc.)
df_raw = df_raw[df_raw["loan_status"].isin(DEFAULT_STATUSES | PAID_STATUSES)].copy()
df_raw[TARGET_COL] = df_raw["loan_status"].isin(DEFAULT_STATUSES).astype(int)
df_raw.drop(columns=["loan_status"], inplace=True)
print(f"After status filter: {df_raw.shape}")
print(f"Default rate: {df_raw[TARGET_COL].mean():.2%}")

# Parse date; sort chronologically
df_raw["issue_d"] = pd.to_datetime(df_raw["issue_d"], format="%b-%Y", errors="coerce")
df_raw = df_raw.dropna(subset=["issue_d"]).sort_values("issue_d").reset_index(drop=True)

# Parse text fields
df_raw["emp_length"] = df_raw["emp_length"].apply(parse_employment_length)
df_raw["revol_util"] = df_raw["revol_util"].apply(parse_revolving_util)
df_raw["term"]       = df_raw["term"].str.strip().str.replace(" months","",regex=False).astype(float)
df_raw["int_rate"]   = pd.to_numeric(df_raw["int_rate"].astype(str).str.replace("%","",regex=False), errors="coerce")

# Add macro features
df_raw = add_macro_features(df_raw)

# Feature engineering
df_raw = engineer_features(df_raw)

# Encode categoricals
df_raw = encode_categoricals(df_raw)

# Drop columns no longer needed after engineering
df_raw.drop(columns=["earliest_cr_line","addr_state"], inplace=True, errors="ignore")

# Impute remaining NaN with column median
# NOTE: `df_raw[col].fillna(..., inplace=True)` is a silent no-op under pandas
# Copy-on-Write (default pandas>=2.0, mandatory pandas>=3.0) because df_raw[col]
# returns a copy, not a view. Must reassign instead.
for col in df_raw.select_dtypes(include=[np.number]).columns:
    if df_raw[col].isna().any():
        df_raw[col] = df_raw[col].fillna(df_raw[col].median())

print(f"Processed shape: {df_raw.shape}")
print(f"Date range: {df_raw['issue_d'].min().date()} → {df_raw['issue_d'].max().date()}")
print(f"Done in {time.time()-t0:.1f}s")


In [ ]:
# ── TEMPORAL TRAIN / TEST SPLIT ──────────────────────────────────
# Train = 2007–2017. Test = 2018.
# This is a strict temporal split — no data leakage across time.

train_df = df_raw[df_raw["issue_d"] <= TRAIN_END_DATE].copy()
test_df  = df_raw[df_raw["issue_d"] >= TEST_START_DATE].copy()

FEATURE_COLS = get_feature_columns(df_raw)

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print(f"Train set: {X_train.shape}  |  Default rate: {y_train.mean():.2%}")
print(f"Test  set: {X_test.shape}   |  Default rate: {y_test.mean():.2%}")
print(f"Features:  {len(FEATURE_COLS)}")

# Save to parquet for fast reloading in downstream cells
train_df.to_parquet(os.path.join(WORKING_DIR, "train.parquet"), index=False)
test_df.to_parquet( os.path.join(WORKING_DIR, "test.parquet"),  index=False)
print("Splits saved to working directory.")


## Cell Cluster 3 — XGBoost Credit Default Model

We use XGBoost as our primary model because:
- TreeSHAP produces **exact** (non-sampled) Shapley values for tree models — essential for XSI computation
- Industry-standard in credit risk; achieves state-of-the-art AUC on tabular loan data
- Handles class imbalance via `scale_pos_weight`
- `tree_method='hist'` is GPU-compatible on Kaggle accelerator instances

Set `N_OPTUNA_TRIALS > 0` to run hyperparameter search. Use `0` for a fast run with defaults.


In [ ]:
# ── METRIC HELPERS ───────────────────────────────────────────────

def ks_statistic(y_true, y_prob):
    """
    KS statistic: separation between default and non-default score distributions.
    KS > 0.40 = strong credit scorecard. Standard metric in credit risk management.
    """
    pos = pd.Series(y_prob)[pd.Series(y_true).values == 1]
    neg = pd.Series(y_prob)[pd.Series(y_true).values == 0]
    ks, _ = ks_2samp(pos, neg)
    return ks

def gini_coefficient(y_true, y_prob):
    """Gini = 2*AUC - 1. Preferred over AUC in credit risk literature."""
    return 2 * roc_auc_score(y_true, y_prob) - 1

# ── OPTUNA OBJECTIVE ──────────────────────────────────────────────

def optuna_objective(trial, X_tr, y_tr):
    """5-fold CV AUC for Optuna hyperparameter search."""
    params = {
        "objective":        "binary:logistic",
        "eval_metric":      "auc",
        "tree_method":      "hist",
        "device":           XGB_PARAMS["device"],
        "n_estimators":     trial.suggest_int("n_estimators", 400, 1200),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.5, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 6.0),
        "random_state":     RANDOM_SEED, "n_jobs": -1,
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, val_idx in cv.split(X_tr, y_tr):
        m = xgb.XGBClassifier(**params, early_stopping_rounds=50, verbosity=0)
        m.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx],
              eval_set=[(X_tr.iloc[val_idx], y_tr.iloc[val_idx])], verbose=False)
        scores.append(roc_auc_score(y_tr.iloc[val_idx], m.predict_proba(X_tr.iloc[val_idx])[:,1]))
    return np.mean(scores)

print("Model helpers defined.")


In [ ]:
# ── TRAIN XGBOOST MODEL ───────────────────────────────────────────
# Set N_OPTUNA_TRIALS = 50 for a proper search (takes ~20 min on Kaggle GPU).
# Set to 0 to use the config defaults and train in ~3 minutes.
N_OPTUNA_TRIALS = 0

best_params = XGB_PARAMS.copy()

if N_OPTUNA_TRIALS > 0:
    print(f"Running Optuna ({N_OPTUNA_TRIALS} trials)...")
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(lambda t: optuna_objective(t, X_train, y_train),
                   n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"Best CV AUC: {study.best_value:.4f}")
    best_params.update(study.best_params)
    with open(os.path.join(MODELS_DIR, "xgb_best_params.json"), "w") as f:
        json.dump({k:v for k,v in best_params.items() if isinstance(v,(int,float,str))}, f, indent=2)

# Final train: 85% for training, 15% for early-stopping validation only
# (proper evaluation uses the held-out temporal test set, not this slice)
val_cut      = int(len(X_train) * 0.85)
X_tr, X_val  = X_train.iloc[:val_cut], X_train.iloc[val_cut:]
y_tr, y_val  = y_train.iloc[:val_cut], y_train.iloc[val_cut:]

print("Training XGBoost model...")
t0 = time.time()
xgb_model = xgb.XGBClassifier(**best_params, early_stopping_rounds=50, verbosity=1)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
print(f"Training complete in {time.time()-t0:.1f}s")

# Save model artifact
with open(os.path.join(MODELS_DIR, "xgboost_model.pkl"), "wb") as f:
    pickle.dump({"model": xgb_model, "feature_cols": FEATURE_COLS}, f)
print("Model saved.")


In [ ]:
# ── FULL EVALUATION ──────────────────────────────────────────────
print("=" * 55)
print("  MODEL EVALUATION")
print("=" * 55)

for split_name, X_eval, y_eval in [
    ("Train 2007-2017", X_train, y_train),
    ("Test  2018",      X_test,  y_test),
]:
    if len(y_eval) == 0:
        print(f"\n  {split_name} — skipped (empty split)")
        continue
    prob = xgb_model.predict_proba(X_eval)[:, 1]
    pred = (prob >= 0.5).astype(int)
    auc  = roc_auc_score(y_eval, prob)
    ks   = ks_statistic(y_eval, prob)
    gini = gini_coefficient(y_eval, prob)
    brier= brier_score_loss(y_eval, prob)
    ap   = average_precision_score(y_eval, prob)

    print(f"\n  {split_name}")
    print(f"    AUC:              {auc:.4f}")
    print(f"    Gini coefficient: {gini:.4f}")
    print(f"    KS statistic:     {ks:.4f}  (>0.40 = strong scorecard)")
    print(f"    Brier score:      {brier:.4f}  (lower is better)")
    print(f"    Avg precision:    {ap:.4f}")
    print(f"\n{classification_report(y_eval, pred, target_names=['Fully Paid','Default'])}")

# ── ROC Curve ─────────────────────────────────────────────────────
prob_test = xgb_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, prob_test)
auc_test    = roc_auc_score(y_test, prob_test)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ROC
axes[0].plot(fpr, tpr, color="royalblue", lw=2, label=f"AUC = {auc_test:.3f}")
axes[0].plot([0,1],[0,1],"k--", lw=1)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve (Test 2018)"); axes[0].legend()

# Calibration
frac_pos, mean_pred = calibration_curve(y_test, prob_test, n_bins=10)
axes[1].plot(mean_pred, frac_pos, "s-", color="royalblue", label="XGBoost")
axes[1].plot([0,1],[0,1],"k--", label="Perfect")
axes[1].set_xlabel("Mean Predicted Prob"); axes[1].set_ylabel("Actual Default Rate")
axes[1].set_title("Calibration Curve"); axes[1].legend()

# Feature importance (gain)
fi = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(20)
sns.barplot(x=fi.values, y=fi.index, ax=axes[2], palette="Blues_r")
axes[2].set_xlabel("Importance"); axes[2].set_title("Top 20 Features (Gain)")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "xgb_evaluation.png"), dpi=300, bbox_inches="tight")
plt.show()


## Cell Cluster 3 — XAI Engine: SHAP and LIME

We use **TreeSHAP** as the primary XAI method:
- Exact (non-sampled) Shapley values for tree models
- Satisfies completeness, symmetry, dummy axioms
- `model_output='probability'` explains predicted default probabilities directly

**LIME** serves as a second-opinion comparator:
- Local linear surrogate fitted around each prediction
- Stochastic: controlled via `num_samples` and `random_state`
- Comparing SHAP vs LIME temporal stability is itself a paper result

We produce: global beeswarm summary, single-loan waterfall, SHAP dependence plots, and LIME local explanations.


In [ ]:
# ══════════════════════════════════════════════════════════════
# SHAP ENGINE
# ══════════════════════════════════════════════════════════════

class SHAPEngine:
    def __init__(self, model, feature_cols, background_data=None):
        self.model        = model
        self.feature_cols = feature_cols
        print("[SHAPEngine] Initializing TreeSHAP explainer...")
        # TreeSHAP with interventional perturbation + probability output.
        # background_data sets the baseline (expected value) for attributions.
        self.explainer = shap.TreeExplainer(
            model,
            data=background_data,
            model_output="probability",
            feature_perturbation="interventional",
        )
        self.expected_value = self.explainer.expected_value
        print(f"[SHAPEngine] Ready. Base rate (E[f(x)]) = {self.expected_value:.4f}")

    def compute_shap_values(self, X, check_additivity=False):
        """Returns SHAP matrix (n_samples, n_features)."""
        X_ = X[self.feature_cols] if isinstance(X, pd.DataFrame) else X
        sv  = self.explainer.shap_values(X_, check_additivity=check_additivity)
        if isinstance(sv, list): sv = sv[1]  # positive class
        return sv

    def get_mean_abs_shap(self, X):
        """Global importance: mean |SHAP| per feature, sorted descending."""
        sv = self.compute_shap_values(X)
        return pd.Series(np.abs(sv).mean(axis=0), index=self.feature_cols).sort_values(ascending=False)

    def get_top_features(self, X, k=None):
        k = k or TOP_K_FEATURES
        return list(self.get_mean_abs_shap(X).head(k).index)

    def get_attribution_vector(self, X):
        """Signed mean attribution vector. Used for XSI cosine similarity."""
        return np.mean(self.compute_shap_values(X), axis=0)

    def plot_summary(self, X, title="SHAP Summary", save_path=None):
        sv = self.compute_shap_values(X[self.feature_cols])
        shap.summary_plot(sv, X[self.feature_cols], feature_names=self.feature_cols,
                          max_display=20, show=False)
        plt.title(title)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show(); plt.close()

    def plot_waterfall(self, X, index=0, title=None, save_path=None):
        """Waterfall plot: shows how each feature pushes one prediction
        away from the base rate. Used for individual loan explanations."""
        X_   = X[self.feature_cols].iloc[[index]]
        sv   = self.compute_shap_values(X_)[0]
        exp  = shap.Explanation(values=sv, base_values=self.expected_value,
                                data=X_.values[0], feature_names=self.feature_cols)
        shap.waterfall_plot(exp, max_display=15, show=False)
        if title: plt.title(title)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show(); plt.close()

print("SHAPEngine class defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# LIME ENGINE
# ══════════════════════════════════════════════════════════════

class LIMEEngine:
    def __init__(self, model, feature_cols, train_data, random_state=42):
        self.model        = model
        self.feature_cols = feature_cols
        print("[LIMEEngine] Initializing LIME tabular explainer...")
        # LIME needs training data to understand feature distributions
        # for its perturbation sampling strategy.
        # Low-cardinality integer columns (binary flags, term, vix_bucket, etc.)
        # break LIME's quantile discretizer (zero-width bins -> scipy truncnorm
        # domain error). Mark them as categorical so LIME samples them directly.
        cat_idx = [i for i, c in enumerate(feature_cols)
                   if train_data[c].nunique() <= 10]
        self.explainer = LimeTabularExplainer(
            training_data=train_data[feature_cols].values,
            feature_names=feature_cols,
            class_names=["Fully Paid", "Default"],
            mode="classification",
            random_state=random_state,
            discretize_continuous=True,
            categorical_features=cat_idx,
        )
        print("[LIMEEngine] Ready.")

    def explain_instance(self, row, num_features=15, num_samples=2000):
        """Local LIME explanation for one row. num_samples controls stability."""
        row_vals = row[self.feature_cols].values.flatten()
        return self.explainer.explain_instance(
            data_row=row_vals,
            predict_fn=lambda x: self.model.predict_proba(
                pd.DataFrame(x, columns=self.feature_cols)),
            num_features=num_features,
            num_samples=num_samples,
            labels=(1,),
        )

    def get_lime_weights(self, row, num_features=15):
        exp = self.explain_instance(row, num_features)
        return pd.Series(dict(exp.as_list(label=1)))

    def get_mean_lime_importance(self, X, n_samples=100, num_features=15):
        """Approximate global importance by averaging LIME weights over n_samples."""
        sample = X.sample(min(n_samples, len(X)), random_state=RANDOM_SEED)
        weights = []
        for _, row in sample.iterrows():
            w = self.get_lime_weights(row.to_frame().T, num_features)
            weights.append(w)
        return pd.DataFrame(weights).fillna(0).abs().mean().sort_values(ascending=False)

    def plot_explanation(self, row, title="LIME Explanation", save_path=None):
        exp = self.explain_instance(row)
        fig = exp.as_pyplot_figure(label=1)
        plt.title(title); plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show(); plt.close()

print("LIMEEngine class defined.")


In [ ]:
# ── RUN SHAP AND LIME EXPLANATIONS ───────────────────────────────
# Sample from test set for tractable compute
sample      = test_df.sample(min(SHAP_EXPLAIN_SAMPLES, len(test_df)), random_state=RANDOM_SEED)
X_sample    = sample[FEATURE_COLS]
bg_data     = X_sample.sample(SHAP_BACKGROUND_SAMPLES, random_state=RANDOM_SEED)

# Initialise engines
shap_engine = SHAPEngine(xgb_model, FEATURE_COLS, background_data=bg_data)
lime_engine = LIMEEngine(xgb_model, FEATURE_COLS, train_df)

# ── Global SHAP beeswarm ──────────────────────────────────────────
print("Generating global SHAP summary...")
shap_engine.plot_summary(
    X_sample,
    title="SHAP Beeswarm — Credit Default Model (Test Set)",
    save_path=os.path.join(FIGURES_DIR, "shap_summary_global.png"),
)

# ── Waterfall for highest-risk loan ──────────────────────────────
probs        = xgb_model.predict_proba(X_sample)[:, 1]
high_risk_i  = int(np.argmax(probs))
print(f"\nHighest-risk loan: index {high_risk_i}, P(default) = {probs[high_risk_i]:.3f}")

shap_engine.plot_waterfall(
    X_sample, index=high_risk_i,
    title=f"SHAP Waterfall — High Risk Loan (P(default)={probs[high_risk_i]:.3f})",
    save_path=os.path.join(FIGURES_DIR, "shap_waterfall_highrisk.png"),
)

# ── LIME local explanation for same loan ─────────────────────────
print("Generating LIME explanation...")
lime_engine.plot_explanation(
    X_sample.iloc[[high_risk_i]],
    title=f"LIME — High Risk Loan (P(default)={probs[high_risk_i]:.3f})",
    save_path=os.path.join(FIGURES_DIR, "lime_explanation_highrisk.png"),
)


In [ ]:
# ── SHAP vs LIME GLOBAL RANKING COMPARISON ───────────────────────
# Compares the two methods' feature importance rankings.
# High Spearman ρ = both methods agree on what matters.
# This is itself reported as a result in the paper.

import re

shap_imp = shap_engine.get_mean_abs_shap(X_sample).head(15)
lime_imp = lime_engine.get_mean_lime_importance(X_sample, n_samples=200, num_features=15)

# LIME returns rule strings like "sub_grade > 15.00" or "0.5 < dti <= 2.0"
# as its keys. We strip these back to raw feature names so they can be
# matched against SHAP's clean feature name index.
def extract_feature_name(lime_key):
    # Split on whitespace, comparators, and digits-only tokens
    # The actual feature name is the first alphabetic token found
    parts = re.split(r'[\s><!=]+', lime_key.strip())
    for part in parts:
        if part and not part.replace('.','').replace('-','').isdigit():
            return part
    return lime_key

lime_imp_clean = pd.Series(
    {extract_feature_name(k): v for k, v in lime_imp.items()}
).groupby(level=0).mean().sort_values(ascending=False)

print("LIME cleaned feature names:", list(lime_imp_clean.index[:10]))
print("SHAP feature names:        ", list(shap_imp.index[:10]))

common = list(set(shap_imp.index) & set(lime_imp_clean.index))
print(f"Common features found: {len(common)} → {common}")

if len(common) < 2:
    print("WARNING: Too few common features to compute correlation.")
else:
    shap_rnk  = shap_imp[common].rank(ascending=False)
    lime_rnk  = lime_imp_clean[common].rank(ascending=False)
    corr, pval = spearmanr(shap_rnk, lime_rnk)
    print(f"SHAP vs LIME Spearman ρ = {corr:.3f}  (p = {pval:.4f})")

    # Side-by-side bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
    for ax, vals, color, label in [
        (axes[0], shap_imp[common].sort_values(ascending=False),       "royalblue", "SHAP (mean |φ|)"),
        (axes[1], lime_imp_clean[common].sort_values(ascending=False),  "tomato",    "LIME (mean |weight|)"),
    ]:
        ax.barh(vals.index[::-1], vals.values[::-1], color=color, alpha=0.85)
        ax.set_xlabel("Importance"); ax.set_title(label)

    fig.suptitle(f"SHAP vs LIME Global Feature Importance\nSpearman ρ = {corr:.3f}", fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "shap_vs_lime_comparison.png"), dpi=300, bbox_inches="tight")
    plt.show()

## Cell Cluster 4 — XDrift Framework (Core Novelty)

This cluster implements the three novel contributions of the paper:

**A. Rolling-Window SHAP** — for each time window, trains a fresh XGBoost model and computes SHAP attributions, capturing how explanations evolve over time.

**B. XSI (Explanation Stability Index)** — a composite metric:
```
XSI_t = 0.4 · Kendall_τ(rank_t, rank_{t-1})    # rank order stability
       + 0.4 · cosine_sim(shap_t, shap_{t-1})   # directional stability
       + 0.2 · (1 - PSI_shap(t, t-1))           # distribution stability
```
XSI ∈ [0, 1]. XSI = 1 means explanations are identical to the previous window.

**C. Regime Detection (HMM)** — 3-state Hidden Markov Model on macroeconomic features labels each window as Bull / Neutral / Bear+Crisis.

**D. EART (Explanation-Aware Retraining Trigger)** — triggers retraining when XSI drops below threshold, *before* AUC degrades. The lead time between EART and AUC-based triggers is a key result.


In [ ]:
# ══════════════════════════════════════════════════════════════
# XSI — EXPLANATION STABILITY INDEX
# ══════════════════════════════════════════════════════════════

def compute_psi(arr_a, arr_b, n_bins=10, eps=1e-6):
    """
    Population Stability Index applied to SHAP value distributions.
    PSI < 0.10 = stable | 0.10-0.25 = some shift | >0.25 = major drift.
    We apply this to SHAP attribution arrays, not just model scores.
    """
    bins = np.percentile(arr_a, np.linspace(0, 100, n_bins + 1))
    bins[0] -= 1e-3; bins[-1] += 1e-3
    a_cnt = np.histogram(arr_a, bins=bins)[0] + eps
    b_cnt = np.histogram(arr_b, bins=bins)[0] + eps
    a_pct = a_cnt / a_cnt.sum(); b_pct = b_cnt / b_cnt.sum()
    return float(np.sum((b_pct - a_pct) * np.log(b_pct / a_pct)))

def compute_xsi(shap_vec_t, shap_vec_t1, rank_vec_t, rank_vec_t1,
                shap_matrix_t=None, shap_matrix_t1=None,
                alpha=0.4, beta=0.4, gamma=0.2):
    """
    Compute XSI between two consecutive windows.

    Component 1 (alpha=0.4): Kendall tau on feature rank vectors.
      Measures whether the relative ordering of feature importances is stable.

    Component 2 (beta=0.4): Cosine similarity of mean SHAP attribution vectors.
      Measures directional consistency (which features push predictions up/down).

    Component 3 (gamma=0.2): 1 - mean PSI across top-K features.
      Measures whether the distribution of SHAP values has shifted.

    Returns xsi in [0,1] and a dict of component values.
    """
    # Component 1: Kendall tau, rescaled from [-1,1] to [0,1]
    tau, _ = kendalltau(rank_vec_t, rank_vec_t1)
    tau_norm = (tau + 1) / 2.0

    # Component 2: Cosine similarity (floored at 0 — negative = fully unstable)
    n_t, n_t1 = np.linalg.norm(shap_vec_t), np.linalg.norm(shap_vec_t1)
    cos_sim = float(np.dot(shap_vec_t, shap_vec_t1) / (n_t * n_t1)) if (n_t > 1e-10 and n_t1 > 1e-10) else 0.0
    cos_sim = max(0.0, cos_sim)

    # Component 3: PSI on SHAP distributions
    psi_component = 0.0
    if shap_matrix_t is not None and shap_matrix_t1 is not None:
        psi_scores = [min(compute_psi(shap_matrix_t[:,k], shap_matrix_t1[:,k]), 1.0)
                      for k in range(min(TOP_K_FEATURES, shap_matrix_t.shape[1]))]
        psi_component = max(0.0, 1.0 - np.mean(psi_scores))
    else:
        psi_component = (tau_norm + cos_sim) / 2  # fallback if no matrices

    xsi = alpha * tau_norm + beta * cos_sim + gamma * psi_component

    return xsi, {
        "kendall_tau":   tau,       "kendall_norm":  tau_norm,
        "cosine_sim":    cos_sim,   "psi_component": psi_component,
        "xsi":           xsi,
    }

print("XSI functions defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# ROLLING WINDOW MANAGER
# ══════════════════════════════════════════════════════════════

class RollingWindowManager:
    """
    Generates rolling time windows over the full loan dataset.
    Each window provides a training slice and a forward eval slice.
    Timeline (WINDOW=12mo, STRIDE=3mo):
      Window 1: train Jan07-Dec07, eval Jan08-Mar08
      Window 2: train Apr07-Mar08, eval Apr08-Jun08  ... etc.
    """
    def __init__(self, df):
        self.df = df.copy()
        self.df[DATE_COL] = pd.to_datetime(self.df[DATE_COL])
        self.df.sort_values(DATE_COL, inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        self.min_date = self.df[DATE_COL].min()
        self.max_date = self.df[DATE_COL].max()
        self.windows  = self._generate_windows()
        print(f"[RollingWindow] {len(self.windows)} windows "
              f"({self.min_date.date()} to {self.max_date.date()})")

    def _generate_windows(self):
        windows = []
        start = self.min_date
        while True:
            end      = start + pd.DateOffset(months=WINDOW_MONTHS)
            eval_end = end   + pd.DateOffset(months=STRIDE_MONTHS)
            if eval_end > self.max_date: break
            windows.append({"train_start":start, "train_end":end,
                             "eval_start":end, "eval_end":eval_end,
                             "label":end.strftime("%Y-%m")})
            start += pd.DateOffset(months=STRIDE_MONTHS)
        return windows

    def get_window_data(self, w):
        """Returns (X_train, y_train, X_eval, y_eval, meta) or None if too small."""
        tr_mask  = (self.df[DATE_COL] >= w["train_start"]) & (self.df[DATE_COL] < w["train_end"])
        ev_mask  = (self.df[DATE_COL] >= w["eval_start"])  & (self.df[DATE_COL] < w["eval_end"])
        tr_df, ev_df = self.df[tr_mask], self.df[ev_mask]
        if len(tr_df) < MIN_WINDOW_SIZE or len(ev_df) < 50: return None
        fc = get_feature_columns(self.df)
        meta = {
            "label":       w["label"],        "train_start": w["train_start"],
            "train_end":   w["train_end"],     "train_size":  len(tr_df),
            "eval_size":   len(ev_df),
            "default_rate":      tr_df[TARGET_COL].mean(),
            "eval_default_rate": ev_df[TARGET_COL].mean(),
            "fed_funds_mean":    tr_df["fed_funds_rate"].mean() if "fed_funds_rate" in tr_df else np.nan,
            "vix_mean":          tr_df["vix_bucket"].mean()     if "vix_bucket"     in tr_df else np.nan,
        }
        return tr_df[fc], tr_df[TARGET_COL], ev_df[fc], ev_df[TARGET_COL], meta

print("RollingWindowManager defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# REGIME DETECTOR (HMM)
# ══════════════════════════════════════════════════════════════

class RegimeDetector:
    """
    Fits a 3-state Gaussian HMM on window-level macroeconomic features
    to classify each window as Bull / Neutral / Bear+Crisis.
    States are post-hoc ordered so state 2 always = highest default rate.
    This lets us stratify XSI by regime — a key paper result.
    """
    def __init__(self):
        self.model  = GaussianHMM(n_components=N_REGIMES, covariance_type="full",
                                   n_iter=200, random_state=RANDOM_SEED)
        self.scaler = StandardScaler()
        self.fitted = False

    def fit(self, results_df):
        feats   = ["fed_funds_mean", "vix_mean", "default_rate"]
        X       = results_df[feats].ffill().fillna(0).values
        X_sc    = self.scaler.fit_transform(X)
        self.model.fit(X_sc)
        self.fitted = True
        print(f"[HMM] Log-likelihood: {self.model.score(X_sc):.2f}")
        return self

    def predict(self, results_df):
        feats   = ["fed_funds_mean", "vix_mean", "default_rate"]
        X       = results_df[feats].ffill().fillna(0).values
        X_sc    = self.scaler.transform(X)
        raw     = self.model.predict(X_sc)
        # Remap states: 0=Bull (lowest default), 1=Neutral, 2=Bear/Crisis (highest)
        regime_dr = {r: results_df.loc[raw==r, "default_rate"].mean()
                     if (raw==r).sum()>0 else 0 for r in range(N_REGIMES)}
        remap = {old: new for new, old in enumerate(sorted(regime_dr, key=lambda r: regime_dr[r]))}
        return np.array([remap[r] for r in raw])

print("RegimeDetector defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# XDRIFT RUNNER — ties everything together
# ══════════════════════════════════════════════════════════════

class XDriftRunner:
    """
    Orchestrates the full XDrift pipeline:
    - Iterates rolling windows
    - Trains a fresh XGBoost per window (fast, 300 estimators)
    - Computes TreeSHAP attribution vectors per window
    - Computes XSI between consecutive windows
    - Stores all per-window metrics
    """
    def __init__(self, df):
        self.df      = df
        self.manager = RollingWindowManager(df)
        self.results = []
        self.feature_cols = None

    def _train_window_model(self, X_tr, y_tr):
        """Lightweight XGBoost for rolling windows (fewer trees = faster)."""
        p = XGB_PARAMS.copy()
        p["n_estimators"] = 300; p["verbosity"] = 0
        p.pop("early_stopping_rounds", None)
        m = xgb.XGBClassifier(**p)
        m.fit(X_tr, y_tr)
        return m

    def run(self, shap_samples=300):
        prev_shap_vec = prev_rank_vec = prev_shap_matrix = None

        for w in tqdm(self.manager.windows, desc="Rolling windows"):
            data = self.manager.get_window_data(w)
            if data is None: continue
            X_tr, y_tr, X_ev, y_ev, meta = data
            self.feature_cols = list(X_tr.columns)

            # Train window model
            model = self._train_window_model(X_tr, y_tr)

            # Eval AUC
            auc = roc_auc_score(y_ev, model.predict_proba(X_ev)[:,1]) \
                  if len(y_ev.unique()) > 1 else np.nan

            # Compute SHAP on eval sample
            n_samp = min(shap_samples, len(X_ev))
            X_samp = X_ev.sample(n_samp, random_state=RANDOM_SEED)
            # Pass a small background sample so interventional perturbation works.
            # We use the training data for this window as the background reference.
            bg = X_tr.sample(min(100, len(X_tr)), random_state=RANDOM_SEED)
            explainer   = shap.TreeExplainer(model, data=bg,
                                              model_output="probability",
                                              feature_perturbation="interventional")
            shap_matrix = explainer.shap_values(X_samp, check_additivity=False)
            if isinstance(shap_matrix, list): shap_matrix = shap_matrix[1]

            shap_vec = np.mean(shap_matrix, axis=0)
            abs_shap = np.abs(shap_vec)
            rank_vec = abs_shap.argsort()[::-1].argsort() + 1  # rank 1 = most important

            # Compute XSI vs previous window
            xsi, components = np.nan, {}
            if prev_shap_vec is not None:
                xsi, components = compute_xsi(
                    shap_vec, prev_shap_vec, rank_vec, prev_rank_vec,
                    shap_matrix, prev_shap_matrix,
                )

            self.results.append({
                "window_label": meta["label"],   "train_start": meta["train_start"],
                "train_end":    meta["train_end"],"train_size":  meta["train_size"],
                "eval_size":    meta["eval_size"],"default_rate":meta["default_rate"],
                "eval_default_rate": meta["eval_default_rate"],
                "fed_funds_mean": meta["fed_funds_mean"], "vix_mean": meta["vix_mean"],
                "eval_auc": auc, "xsi": xsi,
                **{f"xsi_{k}": v for k, v in components.items()},
                "shap_vec": shap_vec, "shap_matrix": shap_matrix, "rank_vec": rank_vec,
                "top_feature": self.feature_cols[int(np.argmax(abs_shap))],
            })
            prev_shap_vec = shap_vec; prev_rank_vec = rank_vec; prev_shap_matrix = shap_matrix

        print(f"[XDrift] {len(self.results)} windows processed.")
        return self

    def get_results_df(self):
        skip = {"shap_vec", "shap_matrix", "rank_vec"}
        return pd.DataFrame([{k:v for k,v in r.items() if k not in skip}
                              for r in self.results])

    def add_regime_labels(self):
        df  = self.get_results_df()
        det = RegimeDetector()
        det.fit(df)
        labels = det.predict(df)
        for i, r in enumerate(self.results):
            r["regime"]      = int(labels[i])
            r["regime_name"] = REGIME_NAMES[int(labels[i])]
        print("[XDrift] Regime labels added.")
        return self

    def apply_eart(self):
        """
        EART — Explanation-Aware Retraining Trigger.
        Baseline strategy: retrain when AUC < 0.70.
        EART strategy:     retrain when XSI < XSI_ALERT_THRESHOLD.
        Key metric: how many windows (months) EART fires BEFORE AUC degradation.
        """
        AUC_THRESHOLD = 0.70
        eart_list, base_list = [], []

        for r in self.results:
            xsi = r.get("xsi", np.nan); auc = r.get("eval_auc", np.nan)
            eart = (not np.isnan(xsi)) and (xsi < XSI_ALERT_THRESHOLD)
            base = (not np.isnan(auc)) and (auc < AUC_THRESHOLD)
            r["eart_trigger"] = eart; r["baseline_trigger"] = base
            eart_list.append(eart); base_list.append(base)

        # Lead time: how many windows does EART precede the next AUC trigger?
        leads = []
        for i, is_e in enumerate(eart_list):
            if is_e:
                nxt = next((j-i for j in range(i+1,len(base_list)) if base_list[j]), None)
                leads.append(nxt)

        valid     = [l for l in leads if l is not None and l > 0]
        mean_lead = np.mean(valid) if valid else 0.0

        self.eart_summary = {
            "eart_triggers":    sum(eart_list),
            "baseline_triggers":sum(base_list),
            "mean_lead_windows":mean_lead,
            "mean_lead_months": mean_lead * STRIDE_MONTHS,
        }
        print(f"[EART] Triggers — EART: {sum(eart_list)}, Baseline: {sum(base_list)}")
        print(f"[EART] Mean lead time: {mean_lead:.1f} windows = {mean_lead*STRIDE_MONTHS:.0f} months")
        return self

print("XDriftRunner defined.")


In [ ]:
# ── RUN THE XDRIFT FRAMEWORK ─────────────────────────────────────
# This is the main computational step. Expect ~20-40 min on Kaggle
# depending on dataset size and number of windows.
# SHAP_SAMPLES_PER_WINDOW controls speed vs. XSI accuracy tradeoff.

SHAP_SAMPLES_PER_WINDOW = 300  # increase to 500 for final paper run

print("Starting XDrift rolling-window analysis...")
t0     = time.time()
runner = XDriftRunner(df_raw)
runner.run(shap_samples=SHAP_SAMPLES_PER_WINDOW)
runner.add_regime_labels()
runner.apply_eart()
print(f"Total time: {(time.time()-t0)/60:.1f} minutes")

# Save results
with open(os.path.join(RESULTS_DIR, "xdrift_runner.pkl"), "wb") as f:
    pickle.dump(runner, f)
print("Runner saved.")

# Flat results table
results_df = runner.get_results_df()
results_df.to_csv(os.path.join(RESULTS_DIR, "xdrift_results.csv"), index=False)
print(f"Results table: {results_df.shape}")
results_df[["window_label","eval_auc","xsi","regime_name","eart_trigger","baseline_trigger"]].tail(20)


In [ ]:
# ── RESULTS SUMMARY TABLE ────────────────────────────────────────
# Mean XSI / AUC / component values stratified by market regime.
# This is the main results table in the paper.

cols  = ["xsi","eval_auc","xsi_kendall_norm","xsi_cosine_sim","xsi_psi_component"]
avail = [c for c in cols if c in results_df.columns]

if "regime_name" in results_df.columns:
    tbl = results_df.groupby("regime_name")[avail].agg(["mean","std"]).round(4)
    print("\nXSI and AUC by Market Regime:")
    print(tbl.to_string())
    tbl.to_csv(os.path.join(RESULTS_DIR, "results_table_by_regime.csv"))

print("\nOverall statistics:")
print(results_df[avail].describe().round(4).to_string())

print(f"\nEART Summary: {runner.eart_summary}")


## Cell Cluster 5 — Paper Figures

All figures referenced in the paper. Each is saved as a high-resolution PNG to `/kaggle/working/figures/`.


In [ ]:
# ── SHARED VISUAL STYLE ──────────────────────────────────────────
REGIME_COLORS = {"Bull":"#2ecc71", "Neutral":"#f39c12", "Bear/Crisis":"#e74c3c"}
REGIME_INT_COLORS = {0:"#2ecc71", 1:"#f39c12", 2:"#e74c3c"}
XSI_LINE   = "#2c3e50"
ALERT_COL  = "#e74c3c"
CRIT_COL   = "#7d0000"

def shade_regimes(ax, df, date_col="train_end"):
    """Shade plot background by market regime (Bull=green, Crisis=red)."""
    if "regime" not in df.columns: return
    dates   = pd.to_datetime(df[date_col])
    regimes = df["regime"].values
    prev_s, prev_r = dates.iloc[0], regimes[0]
    for i in range(1, len(dates)):
        if regimes[i] != prev_r or i == len(dates)-1:
            ax.axvspan(prev_s, dates.iloc[i], alpha=0.12,
                       color=REGIME_INT_COLORS.get(prev_r,"white"), lw=0)
            prev_s, prev_r = dates.iloc[i], regimes[i]

print("Visual style configured.")


In [ ]:
# ── FIG 1: XSI OVER TIME ──────────────────────────────────────────
# Headline figure: XSI timeline with regime shading and threshold markers.
# Shows how explanation stability degrades during crisis regimes.

df_plot = results_df.dropna(subset=["xsi"]).copy()
df_plot["train_end"] = pd.to_datetime(df_plot["train_end"])
df_plot = df_plot.sort_values("train_end")
df_plot["xsi_ma"] = df_plot["xsi"].rolling(3, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 5))
shade_regimes(ax, df_plot)

ax.plot(df_plot["train_end"], df_plot["xsi"],    color=XSI_LINE, alpha=0.3, lw=1.0, label="XSI (raw)")
ax.plot(df_plot["train_end"], df_plot["xsi_ma"], color=XSI_LINE, lw=2.2, label="XSI (3-window MA)")
ax.axhline(XSI_ALERT_THRESHOLD,    color=ALERT_COL, lw=1.5, ls="--", label=f"Alert ({XSI_ALERT_THRESHOLD})")
ax.axhline(XSI_CRITICAL_THRESHOLD, color=CRIT_COL,  lw=1.5, ls=":",  label=f"Critical ({XSI_CRITICAL_THRESHOLD})")

if "eart_trigger" in df_plot:
    trig = df_plot[df_plot["eart_trigger"] == True]
    ax.scatter(trig["train_end"], trig["xsi"], color=ALERT_COL, zorder=5, s=55, marker="v", label="EART trigger")

patches = [mpatches.Patch(color=c, alpha=0.4, label=n) for n, c in REGIME_COLORS.items()]
h, l = ax.get_legend_handles_labels()
ax.legend(h + patches, l + list(REGIME_COLORS.keys()), loc="lower left", fontsize=8, ncol=3)
ax.set_xlabel("Window End Date"); ax.set_ylabel("XSI")
ax.set_title("Fig 1: XSI Over Time — Temporal Stability of Credit Default Explanations")
ax.set_ylim(0, 1.05)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig1_xsi_over_time.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ── FIG 2: XSI BY REGIME (VIOLIN) ────────────────────────────────
# Tests hypothesis: XSI is significantly lower in Bear/Crisis regimes.

df_v = results_df.dropna(subset=["xsi","regime_name"]).copy()
order  = ["Bull","Neutral","Bear/Crisis"]
colors = [REGIME_COLORS[r] for r in order]

fig, ax = plt.subplots(figsize=(9, 6))
sns.violinplot(data=df_v, x="regime_name", y="xsi", order=order,
               palette=colors, inner="box", linewidth=1.5, ax=ax, alpha=0.75)
ax.axhline(XSI_ALERT_THRESHOLD, color=ALERT_COL, lw=1.5, ls="--", label="Alert threshold")
ax.set_xlabel("Market Regime"); ax.set_ylabel("XSI")
ax.set_title("Fig 2: XSI Distribution by Market Regime"); ax.legend()
for i, r in enumerate(order):
    n = (df_v["regime_name"] == r).sum()
    ax.text(i, 0.02, f"n={n}", ha="center", fontsize=9, color="grey")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig2_xsi_by_regime.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ── FIG 3: XSI COMPONENTS STACKED AREA ──────────────────────────
# Decompose XSI into its three weighted components over time.
# Reveals WHICH aspect of stability changes in each regime:
# e.g. cosine_sim drops → magnitude shift; tau_norm drops → rank reorder.

comp_cols = ["xsi_kendall_norm","xsi_cosine_sim","xsi_psi_component"]
df_c = results_df.dropna(subset=comp_cols).copy()
df_c["train_end"] = pd.to_datetime(df_c["train_end"])
df_c = df_c.sort_values("train_end")

fig, ax = plt.subplots(figsize=(14, 5))
shade_regimes(ax, df_c)
ax.stackplot(
    df_c["train_end"],
    df_c["xsi_kendall_norm"] * 0.4,
    df_c["xsi_cosine_sim"]   * 0.4,
    df_c["xsi_psi_component"]* 0.2,
    labels=["Kendall τ (rank, α=0.4)","Cosine sim (direction, β=0.4)","1-PSI (distribution, γ=0.2)"],
    colors=["#3498db","#9b59b6","#1abc9c"], alpha=0.75,
)
ax.set_xlabel("Window End Date"); ax.set_ylabel("Weighted Component Value")
ax.set_title("Fig 3: XSI Components Breakdown Over Time")
ax.legend(loc="lower left", fontsize=9); ax.set_ylim(0, 1.0)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig3_xsi_components.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ── FIG 4: EART vs BASELINE RETRAINING TRIGGERS ──────────────────
# Two-panel figure: XSI (top) and AUC (bottom), with their respective
# triggers marked. Key result: EART fires earlier than AUC degradation.

df_x = results_df.dropna(subset=["xsi"]).copy()
df_a = results_df.dropna(subset=["eval_auc"]).copy()
for d in [df_x, df_a]:
    d["train_end"] = pd.to_datetime(d["train_end"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: XSI
shade_regimes(ax1, df_x.sort_values("train_end"))
ax1.plot(df_x["train_end"], df_x["xsi"], color=XSI_LINE, lw=1.8, label="XSI")
ax1.axhline(XSI_ALERT_THRESHOLD, color=ALERT_COL, lw=1.5, ls="--", label="XSI alert")
if "eart_trigger" in df_x:
    t = df_x[df_x["eart_trigger"]==True]
    ax1.scatter(t["train_end"], t["xsi"], color=ALERT_COL, zorder=5, s=60, marker="v", label="EART trigger")
ax1.set_ylabel("XSI"); ax1.legend(loc="lower left", fontsize=9)
ax1.set_title("Fig 4: EART vs Baseline Retraining Triggers")

# Bottom: AUC
shade_regimes(ax2, df_a.sort_values("train_end"))
ax2.plot(df_a["train_end"], df_a["eval_auc"], color="#8e44ad", lw=1.8, label="Eval AUC")
ax2.axhline(0.70, color="#c0392b", lw=1.5, ls="--", label="AUC threshold (0.70)")
if "baseline_trigger" in df_a:
    b = df_a[df_a["baseline_trigger"]==True]
    ax2.scatter(b["train_end"], b["eval_auc"], color="#c0392b", zorder=5, s=60, marker="x", label="Baseline trigger")
ax2.set_xlabel("Window End Date"); ax2.set_ylabel("AUC")
ax2.legend(loc="lower left", fontsize=9); ax2.set_ylim(0.55, 1.0)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig4_eart_vs_baseline.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ── FIG 5: SHAP ATTRIBUTION HEATMAP ──────────────────────────────
# Heatmap: rows=features, columns=windows, colour=mean|SHAP|.
# A feature whose colour suddenly changes signals explanation drift.
# One of the most visually striking figures in the paper.

windows_with_shap = [r for r in runner.results if r.get("shap_vec") is not None]
if windows_with_shap:
    fc      = runner.feature_cols
    labels  = [r["window_label"] for r in windows_with_shap]
    mat     = np.stack([np.abs(r["shap_vec"]) for r in windows_with_shap])   # (W, F)
    top_idx = mat.mean(axis=0).argsort()[::-1][:TOP_K_FEATURES]
    mat_top = mat[:, top_idx].T
    feat_nm = [fc[i] for i in top_idx]

    fig, ax = plt.subplots(figsize=(max(14, len(labels)*0.3), 8))
    sns.heatmap(mat_top, xticklabels=labels, yticklabels=feat_nm,
                cmap="YlOrRd", ax=ax, linewidths=0.3, linecolor="white",
                cbar_kws={"label": "Mean |SHAP|"})
    ax.set_xlabel("Window (end of training period)"); ax.set_ylabel("Feature")
    ax.set_title(f"Fig 5: SHAP Attribution Heatmap — Top {TOP_K_FEATURES} Features Across Time")
    plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "fig5_shap_heatmap.png"), dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# ── FIG 7: AUC vs XSI SCATTER ────────────────────────────────────
# Do explanation drift and prediction drift co-occur?
# Key insight: windows with low XSI but still acceptable AUC are
# exactly where EART adds value (fires before the baseline would).

df_sc = results_df.dropna(subset=["xsi","eval_auc"]).copy()
fig, ax = plt.subplots(figsize=(8, 6))

if "regime_name" in df_sc.columns:
    for rname, color in REGIME_COLORS.items():
        sub = df_sc[df_sc["regime_name"] == rname]
        if len(sub): ax.scatter(sub["xsi"], sub["eval_auc"], color=color, label=rname, alpha=0.7, s=55)
else:
    ax.scatter(df_sc["xsi"], df_sc["eval_auc"], color="#2c3e50", alpha=0.6, s=55)

corr = df_sc["xsi"].corr(df_sc["eval_auc"])
ax.text(0.05, 0.95, f"Pearson r = {corr:.3f}", transform=ax.transAxes,
        fontsize=10, verticalalignment="top")

ax.axvline(XSI_ALERT_THRESHOLD, color=ALERT_COL, lw=1.5, ls="--", label="XSI alert")
ax.axhline(0.70, color="#8e44ad", lw=1.5, ls="--", label="AUC baseline")
ax.set_xlabel("XSI (Explanation Stability Index)"); ax.set_ylabel("Eval AUC")
ax.set_title("Fig 7: AUC vs XSI — Do Explanation Drift and Prediction Drift Co-occur?")
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig7_auc_vs_xsi.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"Pearson correlation (XSI vs AUC): {corr:.3f}")
print("Note: low-XSI, still-acceptable-AUC windows = EART's early-warning zone.")


In [ ]:
# ── FINAL SUMMARY ────────────────────────────────────────────────
print("=" * 60)
print("  XDRIFT PIPELINE COMPLETE")
print("=" * 60)
print(f"\nFigures saved to:  {FIGURES_DIR}")
print(f"Results saved to:  {RESULTS_DIR}")
print(f"Models saved to:   {MODELS_DIR}")

import os
print("\nFigures generated:")
for f in sorted(os.listdir(FIGURES_DIR)):
    print(f"  {f}")

print(f"\nEART Summary:")
for k, v in runner.eart_summary.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")
